In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
# --- 1. CONFIGURATION & MODEL LOADING ---
MODEL_PATH = 'agritech_plant_disease_90.keras'

# The dictionary must match your folder alphabetical order exactly
class_indices = {
    0: 'Pepper Bell: Bacterial Spot', 1: 'Pepper Bell: Healthy', 
    2: 'Potato: Early Blight', 3: 'Potato: Late Blight', 4: 'Potato: Healthy', 
    5: 'Tomato: Bacterial Spot', 6: 'Tomato: Early Blight', 7: 'Tomato: Late Blight', 
    8: 'Tomato: Leaf Mold', 9: 'Tomato: Septoria Leaf Spot', 
    10: 'Tomato: Spider Mites', 11: 'Tomato: Target Spot', 
    12: 'Tomato: Yellow Leaf Curl Virus', 13: 'Tomato: Mosaic Virus', 14: 'Tomato: Healthy'
}

# Load the trained model
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Model loaded successfully.")
except:
    print("❌ Error: Could not find model file. Check your MODEL_PATH.")

In [ ]:
# --- 2. THE PRO PREDICTION FUNCTION ---
# This version uses TTA (Test Time Augmentation) for higher accuracy
def predict_plant_disease(img_path, num_guesses=5):
    """
    Takes an image path, runs 5 different augmented looks at it,
    and returns the average consensus.
    """
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0
    
    # Augmentation for prediction stability
    datagen = tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    predictions = []
    for _ in range(num_guesses):
        augmented_img = datagen.random_transform(img_array)
        augmented_img = np.expand_dims(augmented_img, axis=0)
        predictions.append(model.predict(augmented_img, verbose=0))
    
    # Average and Extract
    avg_prediction = np.mean(predictions, axis=0)[0]
    top_3_indices = np.argsort(avg_prediction)[-3:][::-1]
    
    # UI Display
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title("Analyzing Leaf...")
    plt.axis('off')
    plt.show()

    print("\n--- 🩺 Diagnosis Results ---")
    for i, idx in enumerate(top_3_indices):
        label = class_indices[idx]
        conf = avg_prediction[idx] * 100
        prefix = "🏆 TOP GUESS:" if i == 0 else f"Rank {i+1}:"
        print(f"{prefix} {label} ({conf:.2f}%)")
    
    if avg_prediction[top_3_indices[0]] < 0.70:
        print("\n⚠️ Note: Confidence is low. Ensure the leaf is centered and well-lit.")


In [ ]:
# --- 3. EXECUTION ---
# Change this path to test any image
test_image = 'c:/Users/Mohsan/Downloads/images (6).jpg'
predict_plant_disease(test_image)